# BiGRU-LSTM

Part of the reference implementation for *Reliability-Aware Model-Dependent Uncertainty Placement under Data Asymmetry for Probabilistic Load Forecasting*. Both uncertainty placements are produced: the modular placement (post-hoc contextual residual quantiles, route 1) and the integrated placement (in-model quantiles, route 2).

Run the cells in order. Outputs go to `outputs/`. 

## 1. Symmetric protocol ($X = X'$)
Training and testing at 5-minute resolution with all channels. Writes `route1` (point + modular quantiles) and `route2` (integrated quantiles) CSVs for Dec 24 and Dec 30, 2024.

In [ ]:
# ---- repository paths -------------------------------------------------
# REPO defaults to the parent of notebooks/ when run locally, or to
# MyDrive/uncertainty-placement on Colab. Override with SEGAN_REPO.
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _default_repo = "/content/drive/MyDrive/uncertainty-placement"
except ImportError:
    _default_repo = os.path.abspath(os.path.join(os.getcwd(), ".."))
REPO     = os.environ.get("SEGAN_REPO", _default_repo)
DATA_DIR = os.environ.get("SEGAN_DATA", os.path.join(REPO, "data"))
OUT_ROOT = os.environ.get("SEGAN_OUT",  os.path.join(REPO, "outputs"))
os.makedirs(OUT_ROOT, exist_ok=True)
# -------------------------------------------------------------------------
# ============================
# BiGRU-LSTM symmetric evaluation at 5-minute multivariate resolution
# FEATURE-ALIGNED + STANDARDIZED ROUTE1 VERSION
#
# Train: V1 only
# Validate: V2 excluding Dec 24 and Dec 30
# Test: Dec 24 and Dec 30 from V2
#
# Route1:
#   - point BiGRU-LSTM trained ONCE on V1 only
#   - forward one-step prediction on validation
#   - NO retraining inside validation
#   - residual buckets built from validation residuals
#
# Route2:
#   - probabilistic BiGRU-LSTM with q10/q50/q90 head
# ============================


# !pip -q install -U tensorflow pandas numpy scikit-learn

import os
import time
import random
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import backend as K

warnings.filterwarnings("ignore")

# =========================
# 0) Reproducibility
# =========================
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# =========================
# 1) Paths
# =========================
V1_PATH = os.path.join(DATA_DIR, "data_v1.0.csv")
V2_PATH = os.path.join(DATA_DIR, "data_v2.0.csv")

OUT_DIR = OUT_ROOT + "/bigrulstm_out_symmetric_5min_route1_standardized"
os.makedirs(OUT_DIR, exist_ok=True)

MERGED_ROUTE1 = os.path.join(OUT_DIR, "merged_route1_point_uq_series_bigrulstm_symmetric_5min.csv")
MERGED_ROUTE2 = os.path.join(OUT_DIR, "merged_route2_predictions_prob_bigrulstm_symmetric_5min.csv")

# =========================
# 2) Timestamp rules
# =========================
V1_START_DATE = "2023-12-01 00:00:00"
V2_START_DATE = "2024-12-01 00:00:00"
SRC_FREQ = "5min"

# =========================
# 3) Forecast settings
# =========================
STEPS_PER_HOUR = 12
HORIZON_HOURS = 24
PRED_LEN = HORIZON_HOURS * STEPS_PER_HOUR   # 288
SEQ_LEN = 288                               # 24h context

Q_LO, Q_MID, Q_HI = 0.10, 0.50, 0.90
QUANTILES = [Q_LO, Q_MID, Q_HI]

TEST_DAYS = [
    pd.Timestamp("2024-12-24"),
    pd.Timestamp("2024-12-30"),
]

VAL_STRIDE = 1   # keep 1 for dense standardized Route1 residuals

BASE_EXOG = [
    "temp_c",
    "voltage_v",
    "amperage_a",
    "generation_w",
    "hour",
    "dayofweek",
    "minute_of_day",
    "slot_5min",
]

FEAT_COLS = (
    BASE_EXOG
    + ["lag_5min", "lag_10min", "lag_1h", "lag_24h"]
    + ["ma_1h", "ma_3h"]
)

# =========================
# 4) Helpers
# =========================
def _find_col(df, candidates):
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]
    for c in df.columns:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    return None

def enforce_non_crossing(q10, q50, q90):
    qs = np.sort(np.vstack([q10, q50, q90]).T, axis=1)
    return qs[:, 0], qs[:, 1], qs[:, 2]

def merge_daily_route1(day_files, out_path):
    parts = []
    for p in day_files:
        df = pd.read_csv(p)
        df.columns = [c.strip() for c in df.columns]
        parts.append(df)
    merged = pd.concat(parts, ignore_index=True)
    merged["timestamp"] = pd.to_datetime(merged["timestamp"], errors="coerce")
    merged = merged.sort_values("timestamp").reset_index(drop=True)
    merged.to_csv(out_path, index=False)
    return merged

def merge_daily_route2(day_files, out_path):
    parts = []
    for p in day_files:
        df = pd.read_csv(p)
        df.columns = [c.strip() for c in df.columns]
        parts.append(df)
    merged = pd.concat(parts, ignore_index=True)
    merged["timestamp"] = pd.to_datetime(merged["timestamp"], errors="coerce")
    merged = merged.sort_values("timestamp").reset_index(drop=True)

    out = pd.DataFrame({
        "timestamp": merged["timestamp"],
        "Actual Power (W)": merged["actual"],
        "q10 (W)": merged["q10"],
        "q50 (W)": merged["q50"],
        "q90 (W)": merged["q90"],
    })
    out.to_csv(out_path, index=False)
    return out

def load_raw_multivar_5min(path, start_date, src_freq="5min"):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    p_col = _find_col(df, ["power (W)", "power", "load", "active power"])
    t_col = _find_col(df, ["temperature (ºC)", "temperature (C)", "temperature", "temp", "air temperature"])
    v_col = _find_col(df, ["voltage (V)", "voltage"])
    a_col = _find_col(df, ["amperage (A)", "amperage", "current"])
    g_col = _find_col(df, ["generation (w)", "generation", "pv generation"])

    required = {
        "power_w": p_col,
        "temp_c": t_col,
        "voltage_v": v_col,
        "amperage_a": a_col,
        "generation_w": g_col,
    }

    missing = [k for k, v in required.items() if v is None]
    if missing:
        raise ValueError(f"Missing required columns in {path}: {missing}. Found {df.columns.tolist()}")

    print(
        f"[{os.path.basename(path)}] "
        f"power='{p_col}', temp='{t_col}', voltage='{v_col}', amperage='{a_col}', generation='{g_col}'"
    )

    df["timestamp"] = pd.date_range(start=start_date, periods=len(df), freq=src_freq)

    for src in [p_col, t_col, v_col, a_col, g_col]:
        df[src] = pd.to_numeric(df[src], errors="coerce")

    out = df.rename(columns={
        p_col: "power_w",
        t_col: "temp_c",
        v_col: "voltage_v",
        a_col: "amperage_a",
        g_col: "generation_w",
    })[["timestamp", "power_w", "temp_c", "voltage_v", "amperage_a", "generation_w"]].copy()

    out["hour"] = out["timestamp"].dt.hour
    out["dayofweek"] = out["timestamp"].dt.dayofweek
    out["minute_of_day"] = out["timestamp"].dt.hour * 60 + out["timestamp"].dt.minute
    out["slot_5min"] = out["minute_of_day"] // 5

    return out

def force_5min_grid(df):
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)

    full_idx = pd.date_range(
        start=df["timestamp"].min(),
        end=df["timestamp"].max(),
        freq="5min"
    )

    df = (
        df.set_index("timestamp")
          .reindex(full_idx)
          .rename_axis("timestamp")
          .reset_index()
    )

    num_cols = [c for c in df.columns if c != "timestamp"]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df[num_cols] = df[num_cols].interpolate(method="linear", limit_direction="both")
    df[num_cols] = df[num_cols].bfill().ffill()

    df["hour"] = df["timestamp"].dt.hour
    df["dayofweek"] = df["timestamp"].dt.dayofweek
    df["minute_of_day"] = df["timestamp"].dt.hour * 60 + df["timestamp"].dt.minute
    df["slot_5min"] = df["minute_of_day"] // 5

    return df

def add_engineered_features_5min(df):
    df = df.copy().sort_values("timestamp").reset_index(drop=True)

    df["lag_5min"] = df["power_w"].shift(1)
    df["lag_10min"] = df["power_w"].shift(2)
    df["lag_1h"] = df["power_w"].shift(12)
    df["lag_24h"] = df["power_w"].shift(288)

    df["ma_1h"] = df["power_w"].shift(1).rolling(window=12, min_periods=1).mean()
    df["ma_3h"] = df["power_w"].shift(1).rolling(window=36, min_periods=1).mean()

    df = df.bfill().ffill()
    return df

def build_sequence_dataset(feature_df, seq_len=24):
    """
    Sequence target = power at the LAST row in the sequence.
    """
    df = feature_df.copy().reset_index(drop=True)

    X = df[FEAT_COLS].to_numpy(dtype=float)
    y = df["power_w"].to_numpy(dtype=float)
    ts = df["timestamp"].to_numpy()

    X_seq, y_seq, ts_seq = [], [], []
    for end_idx in range(seq_len - 1, len(df)):
        start_idx = end_idx - seq_len + 1
        X_seq.append(X[start_idx:end_idx + 1])
        y_seq.append(y[end_idx])
        ts_seq.append(ts[end_idx])

    return np.asarray(X_seq), np.asarray(y_seq).reshape(-1, 1), np.asarray(ts_seq)

def fit_scalers(train_feat_df):
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    scaler_X.fit(train_feat_df[FEAT_COLS].to_numpy(dtype=float))
    scaler_y.fit(train_feat_df[["power_w"]].to_numpy(dtype=float))
    return scaler_X, scaler_y

def scale_sequence_tensor(X_seq, scaler_X):
    n, s, f = X_seq.shape
    X_flat = X_seq.reshape(-1, f)
    X_scaled = scaler_X.transform(X_flat).reshape(n, s, f)
    return X_scaled

def multi_quantile_pinball_loss(y_true, y_pred):
    y_true_tiled = K.tile(y_true, (1, len(QUANTILES)))
    e = y_true_tiled - y_pred
    losses = []
    for i, q in enumerate(QUANTILES):
        qi = tf.constant(q, dtype=y_pred.dtype)
        ei = e[:, i]
        losses.append(K.mean(K.maximum(qi * ei, (qi - 1.0) * ei)))
    return tf.add_n(losses) / len(QUANTILES)

def create_point_bigrulstm(input_shape):
    model = Sequential([
        Bidirectional(GRU(256, return_sequences=True), input_shape=input_shape),
        Dropout(0.3),
        LSTM(128, return_sequences=False),
        Dropout(0.3),
        Dense(1),
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

def create_prob_bigrulstm(input_shape):
    model = Sequential([
        Bidirectional(GRU(256, return_sequences=True), input_shape=input_shape),
        Dropout(0.3),
        LSTM(128, return_sequences=False),
        Dropout(0.3),
        Dense(3),
    ])
    model.compile(optimizer="adam", loss=multi_quantile_pinball_loss)
    return model

def train_point_model(train_feat_df, val_feat_df, epochs=50, batch_size=32):
    scaler_X, scaler_y = fit_scalers(train_feat_df)

    Xtr_seq, ytr_seq, _ = build_sequence_dataset(train_feat_df, seq_len=SEQ_LEN)
    Xva_seq, yva_seq, _ = build_sequence_dataset(val_feat_df, seq_len=SEQ_LEN)

    if len(Xtr_seq) == 0 or len(Xva_seq) == 0:
        raise ValueError("Not enough samples after sequence construction for point BiGRU-LSTM.")

    Xtr_sc = scale_sequence_tensor(Xtr_seq, scaler_X)
    Xva_sc = scale_sequence_tensor(Xva_seq, scaler_X)

    ytr_sc = scaler_y.transform(ytr_seq)
    yva_sc = scaler_y.transform(yva_seq)

    model = create_point_bigrulstm((SEQ_LEN, len(FEAT_COLS)))
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", patience=3, factor=0.5, min_lr=1e-6),
    ]

    hist = model.fit(
        Xtr_sc,
        ytr_sc,
        validation_data=(Xva_sc, yva_sc),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=0,
    )
    return model, scaler_X, scaler_y, hist

def train_prob_model(train_feat_df, val_feat_df, epochs=50, batch_size=32):
    scaler_X, scaler_y = fit_scalers(train_feat_df)

    Xtr_seq, ytr_seq, _ = build_sequence_dataset(train_feat_df, seq_len=SEQ_LEN)
    Xva_seq, yva_seq, _ = build_sequence_dataset(val_feat_df, seq_len=SEQ_LEN)

    if len(Xtr_seq) == 0 or len(Xva_seq) == 0:
        raise ValueError("Not enough samples after sequence construction for probabilistic BiGRU-LSTM.")

    Xtr_sc = scale_sequence_tensor(Xtr_seq, scaler_X)
    Xva_sc = scale_sequence_tensor(Xva_seq, scaler_X)

    ytr_sc = scaler_y.transform(ytr_seq)
    yva_sc = scaler_y.transform(yva_seq)

    model = create_prob_bigrulstm((SEQ_LEN, len(FEAT_COLS)))
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", patience=3, factor=0.5, min_lr=1e-6),
    ]

    hist = model.fit(
        Xtr_sc,
        ytr_sc,
        validation_data=(Xva_sc, yva_sc),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=0,
    )
    return model, scaler_X, scaler_y, hist

def apply_route1_buckets_5min(test_timestamps, point_pred, res_q):
    tmp = pd.DataFrame({"timestamp": pd.to_datetime(test_timestamps)})
    tmp["slot_5min"] = tmp["timestamp"].dt.hour * 12 + (tmp["timestamp"].dt.minute // 5)
    tmp["dayofweek"] = tmp["timestamp"].dt.dayofweek

    tmp = tmp.merge(res_q, on=["slot_5min", "dayofweek"], how="left")
    tmp[["rq10", "rq50", "rq90"]] = tmp[["rq10", "rq50", "rq90"]].fillna(0.0)

    p10 = point_pred + tmp["rq10"].to_numpy()
    p50 = point_pred + tmp["rq50"].to_numpy()
    p90 = point_pred + tmp["rq90"].to_numpy()

    return enforce_non_crossing(p10, p50, p90)

def make_row_for_timestamp_5min(ts, exog_row, history_power):
    if len(history_power) < 288:
        raise ValueError("Insufficient history_power length for 24h lag.")

    ts = pd.Timestamp(ts)
    minute_of_day = ts.hour * 60 + ts.minute
    slot_5min = minute_of_day // 5

    row = {
        "timestamp": ts,
        "temp_c": float(exog_row["temp_c"]),
        "voltage_v": float(exog_row["voltage_v"]),
        "amperage_a": float(exog_row["amperage_a"]),
        "generation_w": float(exog_row["generation_w"]),
        "hour": int(ts.hour),
        "dayofweek": int(ts.dayofweek),
        "minute_of_day": int(minute_of_day),
        "slot_5min": int(slot_5min),
        "power_w": np.nan,
        "lag_5min": float(history_power[-1]),
        "lag_10min": float(history_power[-2]),
        "lag_1h": float(history_power[-12]),
        "lag_24h": float(history_power[-288]),
        "ma_1h": float(np.mean(history_power[-12:])),
        "ma_3h": float(np.mean(history_power[-36:])),
    }
    return row

def recursive_forecast_point_5min(model, scaler_X, scaler_y, hist_feat_df, eval_day_df):
    context_rows = hist_feat_df.tail(SEQ_LEN).copy()
    if len(context_rows) < SEQ_LEN:
        raise ValueError("Not enough context rows for recursive point rollout.")

    sequence_rows = context_rows.to_dict("records")
    history_power = hist_feat_df["power_w"].tolist()

    preds = []
    for _, cur in eval_day_df.iterrows():
        new_row = make_row_for_timestamp_5min(cur["timestamp"], cur, history_power)

        seq_candidate = pd.DataFrame(sequence_rows[-(SEQ_LEN - 1):] + [new_row])
        X_seq = seq_candidate[FEAT_COLS].to_numpy(dtype=float).reshape(1, SEQ_LEN, len(FEAT_COLS))
        X_sc = scale_sequence_tensor(X_seq, scaler_X)

        yhat_sc = model.predict(X_sc, verbose=0)
        yhat = float(scaler_y.inverse_transform(yhat_sc)[0, 0])

        new_row["power_w"] = yhat
        preds.append(yhat)

        history_power.append(yhat)
        sequence_rows.append(new_row)

    return np.asarray(preds, dtype=float)

def recursive_forecast_prob_5min(model, scaler_X, scaler_y, hist_feat_df, eval_day_df):
    context_rows = hist_feat_df.tail(SEQ_LEN).copy()
    if len(context_rows) < SEQ_LEN:
        raise ValueError("Not enough context rows for recursive probabilistic rollout.")

    sequence_rows = context_rows.to_dict("records")
    history_power = hist_feat_df["power_w"].tolist()

    q10_list, q50_list, q90_list = [], [], []
    for _, cur in eval_day_df.iterrows():
        new_row = make_row_for_timestamp_5min(cur["timestamp"], cur, history_power)

        seq_candidate = pd.DataFrame(sequence_rows[-(SEQ_LEN - 1):] + [new_row])
        X_seq = seq_candidate[FEAT_COLS].to_numpy(dtype=float).reshape(1, SEQ_LEN, len(FEAT_COLS))
        X_sc = scale_sequence_tensor(X_seq, scaler_X)

        yq_sc = model.predict(X_sc, verbose=0)
        yq_sc = np.sort(yq_sc, axis=1)

        q10 = float(scaler_y.inverse_transform(yq_sc[:, [0]])[0, 0])
        q50 = float(scaler_y.inverse_transform(yq_sc[:, [1]])[0, 0])
        q90 = float(scaler_y.inverse_transform(yq_sc[:, [2]])[0, 0])

        aa, bb, cc = enforce_non_crossing(np.array([q10]), np.array([q50]), np.array([q90]))
        q10, q50, q90 = float(aa[0]), float(bb[0]), float(cc[0])

        q10_list.append(q10)
        q50_list.append(q50)
        q90_list.append(q90)

        new_row["power_w"] = q50
        history_power.append(q50)
        sequence_rows.append(new_row)

    return np.asarray(q10_list), np.asarray(q50_list), np.asarray(q90_list)

def forward_one_step_validation_predictions(model, scaler_X, scaler_y, train_df, val_df, stride=1):
    """
    Standardized Route1:
    - use the main point model trained on V1 only
    - run forward one-step prediction over validation
    - update history with ACTUAL validation observation
    - no retraining inside validation
    """
    hist_df = train_df.copy().sort_values("timestamp").reset_index(drop=True)
    val_df = val_df.copy().sort_values("timestamp").reset_index(drop=True)

    if len(hist_df) < max(SEQ_LEN, 288):
        raise ValueError("Need enough pre-validation history for recursive validation rollout.")

    context_rows = hist_df.tail(SEQ_LEN).copy()
    sequence_rows = context_rows.to_dict("records")
    history_power = hist_df["power_w"].tolist()

    preds = []
    actuals = []
    ts_list = []

    for j, (_, cur) in enumerate(val_df.iterrows()):
        if j % 1000 == 0:
            print(f"  validation rolling progress: {j}/{len(val_df)}")

        new_row = make_row_for_timestamp_5min(cur["timestamp"], cur, history_power)

        seq_candidate = pd.DataFrame(sequence_rows[-(SEQ_LEN - 1):] + [new_row])
        X_seq = seq_candidate[FEAT_COLS].to_numpy(dtype=float).reshape(1, SEQ_LEN, len(FEAT_COLS))
        X_sc = scale_sequence_tensor(X_seq, scaler_X)

        yhat_sc = model.predict(X_sc, verbose=0)
        yhat = float(scaler_y.inverse_transform(yhat_sc)[0, 0])
        ytrue = float(cur["power_w"])

        if (j % stride) == 0:
            preds.append(yhat)
            actuals.append(ytrue)
            ts_list.append(pd.Timestamp(cur["timestamp"]))

        # update with ACTUAL validation observation
        new_row["power_w"] = ytrue
        history_power.append(ytrue)
        sequence_rows.append(new_row)

    out = pd.DataFrame({
        "timestamp": ts_list,
        "actual": actuals,
        "pred": preds
    })
    return out

def build_route1_residual_buckets_from_val(model, scaler_X, scaler_y, train_df, val_df):
    df_cal = forward_one_step_validation_predictions(model, scaler_X, scaler_y, train_df, val_df, stride=VAL_STRIDE)

    if df_cal.empty:
        raise ValueError("Validation predictions are empty.")

    df_cal["residual"] = df_cal["actual"] - df_cal["pred"]

    clip_val = np.percentile(np.abs(df_cal["residual"].to_numpy()), 99)
    df_cal["residual"] = np.clip(df_cal["residual"], -clip_val, clip_val)

    df_cal["slot_5min"] = df_cal["timestamp"].dt.hour * 12 + (df_cal["timestamp"].dt.minute // 5)
    df_cal["dayofweek"] = df_cal["timestamp"].dt.dayofweek

    res_q = (
        df_cal.groupby(["slot_5min", "dayofweek"])["residual"]
              .quantile([Q_LO, Q_MID, Q_HI])
              .unstack()
              .rename(columns={Q_LO: "rq10", Q_MID: "rq50", Q_HI: "rq90"})
              .reset_index()
    )

    sigma_global = float(np.std(df_cal["residual"].to_numpy()))
    sigma_global = max(sigma_global, 1e-6)

    return res_q, sigma_global, df_cal

# =========================
# 5) Load V1 and V2 at 5-minute resolution
# =========================
print("Loading V1 and V2 at 5-minute resolution...")
v1_5m = load_raw_multivar_5min(V1_PATH, V1_START_DATE, SRC_FREQ)
v2_5m = load_raw_multivar_5min(V2_PATH, V2_START_DATE, SRC_FREQ)

full_5m = pd.concat([v1_5m, v2_5m], ignore_index=True)
full_5m = full_5m.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)
full_5m = force_5min_grid(full_5m)
full_5m = add_engineered_features_5min(full_5m)

v1_5m = full_5m[full_5m["timestamp"] < pd.Timestamp(V2_START_DATE)].copy().reset_index(drop=True)
v2_5m = full_5m[full_5m["timestamp"] >= pd.Timestamp(V2_START_DATE)].copy().reset_index(drop=True)

print("✅ V1 5-min:", len(v1_5m))
print("✅ V2 5-min:", len(v2_5m))

# =========================
# 6) Symmetric split
# =========================
test_mask = v2_5m["timestamp"].dt.normalize().isin(TEST_DAYS)

train_df = v1_5m.copy().reset_index(drop=True)
val_df = v2_5m[~test_mask].copy().reset_index(drop=True)
test_df_all = v2_5m[test_mask].copy().reset_index(drop=True)

print("✅ Train rows:", len(train_df))
print("✅ Val rows  :", len(val_df))
print("✅ Test rows :", len(test_df_all))

for d in TEST_DAYS:
    cur = test_df_all[test_df_all["timestamp"].dt.normalize() == d].copy()
    if len(cur) != PRED_LEN:
        raise ValueError(f"Test day {d.date()} has {len(cur)} rows, expected {PRED_LEN}.")

# =========================
# Skip rerun if outputs exist
# =========================
if os.path.exists(MERGED_ROUTE1) and os.path.exists(MERGED_ROUTE2):
    print("\n✅ Final merged CSVs already exist. Skipping full pipeline.")
    print("Route1:", MERGED_ROUTE1)
    print("Route2:", MERGED_ROUTE2)

else:
    # =========================
    # 7) Train BiGRU-LSTM models on V1 only
    # =========================
    split_train = int(0.90 * len(train_df))
    tr_df = train_df.iloc[:split_train].copy()
    va_df = train_df.iloc[split_train:].copy()

    if len(va_df) < max(SEQ_LEN + 24, 288):
        va_df = train_df.iloc[-max(SEQ_LEN + 288, 576):].copy()
        tr_df = train_df.iloc[:-len(va_df)].copy()

    print("\nTraining Route1 point BiGRU-LSTM on V1 only...")
    t0 = time.time()
    pt_model, pt_sX, pt_sY, _ = train_point_model(tr_df, va_df)
    print(f"Point model fit time: {time.time()-t0:.2f}s | train_rows={len(tr_df)} | val_rows={len(va_df)}")

    print("\nTraining Route2 probabilistic BiGRU-LSTM on V1 only...")
    t0 = time.time()
    pr_model, pr_sX, pr_sY, _ = train_prob_model(tr_df, va_df)
    print(f"Prob model fit time: {time.time()-t0:.2f}s | train_rows={len(tr_df)} | val_rows={len(va_df)}")

    # =========================
    # 8) Build Route1 residual buckets from standardized forward validation
    # =========================
    print("\nBuilding Route1 residual buckets from validation using forward one-step prediction...")
    res_q, sigma_global, df_res = build_route1_residual_buckets_from_val(
        pt_model, pt_sX, pt_sY, train_df, val_df
    )

    resq_path = os.path.join(OUT_DIR, "bigrulstm_route1_residual_quantiles_from_v2val_5min.csv")
    resid_path = os.path.join(OUT_DIR, "bigrulstm_route1_residuals_from_v2val_5min.csv")
    res_q.to_csv(resq_path, index=False)
    df_res.to_csv(resid_path, index=False)

    print("✅ Saved residual quantiles:", resq_path)
    print("✅ Saved residual table    :", resid_path)
    print(f"Fallback sigma_global: {sigma_global:.4f}")

    # =========================
    # 9) Test on Dec 24 and Dec 30
    # =========================
    r1_day_files = []
    r2_day_files = []

    for k, day_start in enumerate(TEST_DAYS, start=1):
        print("\n" + "=" * 80)
        print(f"Test day {k}: {day_start.date()}")

        day_end = day_start + pd.Timedelta(days=1)
        eval_day_df = test_df_all[
            (test_df_all["timestamp"] >= day_start) &
            (test_df_all["timestamp"] < day_end)
        ].copy().reset_index(drop=True)

        y_true = eval_day_df["power_w"].to_numpy(dtype=float)

        hist_feat_df = full_5m[full_5m["timestamp"] < day_start].copy().reset_index(drop=True)

        # Route1
        point_pred = recursive_forecast_point_5min(
            pt_model, pt_sX, pt_sY, hist_feat_df, eval_day_df
        )
        p10, p50, p90 = apply_route1_buckets_5min(
            eval_day_df["timestamp"].values, point_pred, res_q
        )

        out_r1 = pd.DataFrame({
            "timestamp": eval_day_df["timestamp"].values,
            "actual": y_true,
            "point_pred": point_pred,
            "q10": p10,
            "q50": p50,
            "q90": p90
        })
        p_r1 = os.path.join(OUT_DIR, f"bigrulstm_route1_uq_symmetric_5min_day{k}.csv")
        out_r1.to_csv(p_r1, index=False)
        r1_day_files.append(p_r1)

        # Route2
        q10, q50, q90 = recursive_forecast_prob_5min(
            pr_model, pr_sX, pr_sY, hist_feat_df, eval_day_df
        )

        out_r2 = pd.DataFrame({
            "timestamp": eval_day_df["timestamp"].values,
            "actual": y_true,
            "q10": q10,
            "q50": q50,
            "q90": q90
        })
        p_r2 = os.path.join(OUT_DIR, f"bigrulstm_route2_prob_symmetric_5min_day{k}.csv")
        out_r2.to_csv(p_r2, index=False)
        r2_day_files.append(p_r2)

        print("Saved:")
        print("  Route1_UQ  :", p_r1)
        print("  Route2_Prob:", p_r2)

    # =========================
    # 10) Merge outputs
    # =========================
    merged_route1_df = merge_daily_route1(r1_day_files, MERGED_ROUTE1)
    merged_route2_df = merge_daily_route2(r2_day_files, MERGED_ROUTE2)

    print("\nMerged files written:")
    print("  Route1:", MERGED_ROUTE1)
    print("  Route2:", MERGED_ROUTE2)
    print("  OUT_DIR:", OUT_DIR)

    print("\nRoute1 head:")
    print(merged_route1_df.head())

    print("\nRoute2 head:")
    print(merged_route2_df.head())

## 2. Asymmetric protocol ($X \neq X'$)
Training on the full multivariate history; deployment on hourly data with temperature and calendar features only. Voltage, current and PV generation are filled with hour-of-day means from $D_1$.

In [ ]:
# ---- repository paths -------------------------------------------------
# REPO defaults to the parent of notebooks/ when run locally, or to
# MyDrive/uncertainty-placement on Colab. Override with SEGAN_REPO.
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _default_repo = "/content/drive/MyDrive/uncertainty-placement"
except ImportError:
    _default_repo = os.path.abspath(os.path.join(os.getcwd(), ".."))
REPO     = os.environ.get("SEGAN_REPO", _default_repo)
DATA_DIR = os.environ.get("SEGAN_DATA", os.path.join(REPO, "data"))
OUT_ROOT = os.environ.get("SEGAN_OUT",  os.path.join(REPO, "outputs"))
os.makedirs(OUT_ROOT, exist_ok=True)
# -------------------------------------------------------------------------
# ============================
# BiGRU-LSTM ASYMMETRIC evaluation
#
# Train: V1 only, 5-minute, multivariate
# Validate: V2 excluding Dec 24 and Dec 30
# Test: Dec 24 and Dec 30 from V2, converted to HOURLY
#
# Asymmetry:
#   - Train on rich 5-minute multivariate features
#   - Deploy on hourly temperature + time only
#
# Route1: point BiGRU-LSTM + post-hoc residual buckets
#         residual buckets are built from V2 validation
#         BUT under deployment-style hourly degraded inference
#
# Route2: integrated probabilistic BiGRU-LSTM with q10/q50/q90 head
#         trained on V1 rich regime, inferred on degraded hourly regime
# ============================


# !pip -q install -U tensorflow pandas numpy scikit-learn

import os
import time
import random
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import backend as K

warnings.filterwarnings("ignore")

# =========================
# 0) Reproducibility
# =========================
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# =========================
# 1) Paths
# =========================
V1_PATH = os.path.join(DATA_DIR, "data_v1.0.csv")
V2_PATH = os.path.join(DATA_DIR, "data_v2.0.csv")

OUT_DIR = OUT_ROOT + "/bigrulstm_out_asymmetric_train5min_testhourly"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# 2) Timestamp rules
# =========================
V1_START_DATE = "2023-12-01 00:00:00"
V2_START_DATE = "2024-12-01 00:00:00"
SRC_FREQ = "5min"

# =========================
# 3) Forecast settings
# =========================
SEQ_LEN = 288   # training sequences at 5-min resolution
Q_LO, Q_MID, Q_HI = 0.10, 0.50, 0.90
QUANTILES = [Q_LO, Q_MID, Q_HI]

TEST_DAYS = [
    pd.Timestamp("2024-12-24"),
    pd.Timestamp("2024-12-30"),
]

EXPECTED_HOURLY_POINTS_PER_DAY = 24

TRAIN_FEATURES = [
    "temp_c",
    "voltage_v",
    "amperage_a",
    "generation_w",
    "hour",
    "dayofweek",
    "minute_of_day",
    "slot_5min",
    "lag_5min",
    "lag_10min",
    "lag_1h",
    "lag_24h",
    "ma_1h",
    "ma_3h",
]

# =========================
# 4) Helpers
# =========================
def _find_col(df, candidates):
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]
    for c in df.columns:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    return None

def enforce_non_crossing(q10, q50, q90):
    qs = np.sort(np.vstack([q10, q50, q90]).T, axis=1)
    return qs[:, 0], qs[:, 1], qs[:, 2]

def merge_daily_route1(day_files, out_path):
    parts = []
    for p in day_files:
        df = pd.read_csv(p)
        df.columns = [c.strip() for c in df.columns]
        parts.append(df)
    merged = pd.concat(parts, ignore_index=True)
    merged["timestamp"] = pd.to_datetime(merged["timestamp"], errors="coerce")
    merged = merged.sort_values("timestamp").reset_index(drop=True)
    merged.to_csv(out_path, index=False)
    return merged

def merge_daily_route2(day_files, out_path):
    parts = []
    for p in day_files:
        df = pd.read_csv(p)
        df.columns = [c.strip() for c in df.columns]
        parts.append(df)
    merged = pd.concat(parts, ignore_index=True)
    merged["timestamp"] = pd.to_datetime(merged["timestamp"], errors="coerce")
    merged = merged.sort_values("timestamp").reset_index(drop=True)

    out = pd.DataFrame({
        "timestamp": merged["timestamp"],
        "Actual Power (W)": merged["actual"],
        "q10 (W)": merged["q10"],
        "q50 (W)": merged["q50"],
        "q90 (W)": merged["q90"],
    })
    out.to_csv(out_path, index=False)
    return out

def load_raw_multivar_5min(path, start_date, src_freq="5min"):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    p_col = _find_col(df, ["power (W)", "power", "load", "active power"])
    t_col = _find_col(df, ["temperature (ºC)", "temperature (C)", "temperature", "temp", "air temperature"])
    v_col = _find_col(df, ["voltage (V)", "voltage"])
    a_col = _find_col(df, ["amperage (A)", "amperage", "current"])
    g_col = _find_col(df, ["generation (w)", "generation", "pv generation"])

    required = {
        "power_w": p_col,
        "temp_c": t_col,
        "voltage_v": v_col,
        "amperage_a": a_col,
        "generation_w": g_col,
    }

    missing = [k for k, v in required.items() if v is None]
    if missing:
        raise ValueError(f"Missing required columns in {path}: {missing}. Found {df.columns.tolist()}")

    print(
        f"[{os.path.basename(path)}] "
        f"power='{p_col}', temp='{t_col}', voltage='{v_col}', amperage='{a_col}', generation='{g_col}'"
    )

    df["timestamp"] = pd.date_range(start=start_date, periods=len(df), freq=src_freq)

    for src in [p_col, t_col, v_col, a_col, g_col]:
        df[src] = pd.to_numeric(df[src], errors="coerce")

    out = df.rename(columns={
        p_col: "power_w",
        t_col: "temp_c",
        v_col: "voltage_v",
        a_col: "amperage_a",
        g_col: "generation_w",
    })[["timestamp", "power_w", "temp_c", "voltage_v", "amperage_a", "generation_w"]].copy()

    out["hour"] = out["timestamp"].dt.hour
    out["dayofweek"] = out["timestamp"].dt.dayofweek
    out["minute_of_day"] = out["timestamp"].dt.hour * 60 + out["timestamp"].dt.minute
    out["slot_5min"] = out["minute_of_day"] // 5

    return out

def force_5min_grid(df):
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)

    full_idx = pd.date_range(
        start=df["timestamp"].min(),
        end=df["timestamp"].max(),
        freq="5min"
    )

    df = (
        df.set_index("timestamp")
          .reindex(full_idx)
          .rename_axis("timestamp")
          .reset_index()
    )

    num_cols = [c for c in df.columns if c != "timestamp"]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df[num_cols] = df[num_cols].interpolate(method="linear", limit_direction="both")
    df[num_cols] = df[num_cols].bfill().ffill()

    df["hour"] = df["timestamp"].dt.hour
    df["dayofweek"] = df["timestamp"].dt.dayofweek
    df["minute_of_day"] = df["timestamp"].dt.hour * 60 + df["timestamp"].dt.minute
    df["slot_5min"] = df["minute_of_day"] // 5

    return df

def add_engineered_features_5min(df):
    df = df.copy().sort_values("timestamp").reset_index(drop=True)

    df["lag_5min"] = df["power_w"].shift(1)
    df["lag_10min"] = df["power_w"].shift(2)
    df["lag_1h"] = df["power_w"].shift(12)
    df["lag_24h"] = df["power_w"].shift(288)

    df["ma_1h"] = df["power_w"].shift(1).rolling(window=12, min_periods=1).mean()
    df["ma_3h"] = df["power_w"].shift(1).rolling(window=36, min_periods=1).mean()

    df = df.bfill().ffill()
    return df

def build_sequence_dataset(feature_df, seq_len=24):
    df = feature_df.copy().reset_index(drop=True)

    X = df[TRAIN_FEATURES].to_numpy(dtype=float)
    y = df["power_w"].to_numpy(dtype=float).reshape(-1, 1)
    ts = df["timestamp"].to_numpy()

    X_seq, y_seq, ts_seq = [], [], []
    for end_idx in range(seq_len - 1, len(df)):
        start_idx = end_idx - seq_len + 1
        X_seq.append(X[start_idx:end_idx + 1])
        y_seq.append(y[end_idx])
        ts_seq.append(ts[end_idx])

    return np.asarray(X_seq), np.asarray(y_seq).reshape(-1, 1), np.asarray(ts_seq)

def fit_scalers(train_feat_df):
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    scaler_X.fit(train_feat_df[TRAIN_FEATURES].to_numpy(dtype=float))
    scaler_y.fit(train_feat_df[["power_w"]].to_numpy(dtype=float))
    return scaler_X, scaler_y

def scale_sequence_tensor(X_seq, scaler_X):
    n, s, f = X_seq.shape
    X_flat = X_seq.reshape(-1, f)
    X_scaled = scaler_X.transform(X_flat).reshape(n, s, f)
    return X_scaled

def multi_quantile_pinball_loss(y_true, y_pred):
    y_true_tiled = K.tile(y_true, (1, len(QUANTILES)))
    e = y_true_tiled - y_pred
    losses = []
    for i, q in enumerate(QUANTILES):
        qi = tf.constant(q, dtype=y_pred.dtype)
        ei = e[:, i]
        losses.append(K.mean(K.maximum(qi * ei, (qi - 1.0) * ei)))
    return tf.add_n(losses) / len(QUANTILES)

def create_point_bigrulstm(input_shape):
    model = Sequential([
        Bidirectional(GRU(256, return_sequences=True), input_shape=input_shape),
        Dropout(0.3),
        LSTM(128, return_sequences=False),
        Dropout(0.3),
        Dense(1),
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

def create_prob_bigrulstm(input_shape):
    model = Sequential([
        Bidirectional(GRU(256, return_sequences=True), input_shape=input_shape),
        Dropout(0.3),
        LSTM(128, return_sequences=False),
        Dropout(0.3),
        Dense(3),
    ])
    model.compile(optimizer="adam", loss=multi_quantile_pinball_loss)
    return model

def train_point_model(train_feat_df, val_feat_df, epochs=50, batch_size=32):
    scaler_X, scaler_y = fit_scalers(train_feat_df)

    Xtr_seq, ytr_seq, _ = build_sequence_dataset(train_feat_df, seq_len=SEQ_LEN)
    Xva_seq, yva_seq, _ = build_sequence_dataset(val_feat_df, seq_len=SEQ_LEN)

    if len(Xtr_seq) == 0 or len(Xva_seq) == 0:
        raise ValueError("Not enough samples after sequence construction for point BiGRU-LSTM.")

    Xtr_sc = scale_sequence_tensor(Xtr_seq, scaler_X)
    Xva_sc = scale_sequence_tensor(Xva_seq, scaler_X)

    ytr_sc = scaler_y.transform(ytr_seq)
    yva_sc = scaler_y.transform(yva_seq)

    model = create_point_bigrulstm((SEQ_LEN, len(TRAIN_FEATURES)))
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", patience=3, factor=0.5, min_lr=1e-6),
    ]

    hist = model.fit(
        Xtr_sc,
        ytr_sc,
        validation_data=(Xva_sc, yva_sc),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=0,
    )
    return model, scaler_X, scaler_y, hist

def train_prob_model(train_feat_df, val_feat_df, epochs=50, batch_size=32):
    scaler_X, scaler_y = fit_scalers(train_feat_df)

    Xtr_seq, ytr_seq, _ = build_sequence_dataset(train_feat_df, seq_len=SEQ_LEN)
    Xva_seq, yva_seq, _ = build_sequence_dataset(val_feat_df, seq_len=SEQ_LEN)

    if len(Xtr_seq) == 0 or len(Xva_seq) == 0:
        raise ValueError("Not enough samples after sequence construction for probabilistic BiGRU-LSTM.")

    Xtr_sc = scale_sequence_tensor(Xtr_seq, scaler_X)
    Xva_sc = scale_sequence_tensor(Xva_seq, scaler_X)

    ytr_sc = scaler_y.transform(ytr_seq)
    yva_sc = scaler_y.transform(yva_seq)

    model = create_prob_bigrulstm((SEQ_LEN, len(TRAIN_FEATURES)))
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", patience=3, factor=0.5, min_lr=1e-6),
    ]

    hist = model.fit(
        Xtr_sc,
        ytr_sc,
        validation_data=(Xva_sc, yva_sc),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=0,
    )
    return model, scaler_X, scaler_y, hist

def aggregate_to_hourly(df_5m):
    df = df_5m.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["ts_hour"] = df["timestamp"].dt.floor("H")

    ag = (
        df.groupby("ts_hour", as_index=False)
          .agg({
              "power_w": "mean",
              "temp_c": "mean",
              "voltage_v": "mean",
              "amperage_a": "mean",
              "generation_w": "mean",
          })
          .rename(columns={"ts_hour": "timestamp"})
    )

    ag["hour"] = ag["timestamp"].dt.hour
    ag["dayofweek"] = ag["timestamp"].dt.dayofweek
    ag["minute_of_day"] = ag["timestamp"].dt.hour * 60 + ag["timestamp"].dt.minute
    ag["slot_5min"] = ag["minute_of_day"] // 5

    return ag

def build_hourly_priors_from_train(train_5m):
    train_h = aggregate_to_hourly(train_5m)

    priors_hour = (
        train_h.groupby("hour")[["voltage_v", "amperage_a", "generation_w"]]
               .median()
               .reset_index()
    )

    priors_global = {
        "voltage_v": float(train_h["voltage_v"].median()),
        "amperage_a": float(train_h["amperage_a"].median()),
        "generation_w": float(train_h["generation_w"].median()),
        "lag_5min": float(train_5m["power_w"].shift(1).median()),
        "lag_10min": float(train_5m["power_w"].shift(2).median()),
        "lag_1h": float(train_5m["power_w"].shift(12).median()),
        "lag_24h": float(train_5m["power_w"].shift(288).median()),
        "ma_1h": float(train_5m["power_w"].shift(1).rolling(12, min_periods=1).mean().median()),
        "ma_3h": float(train_5m["power_w"].shift(1).rolling(36, min_periods=1).mean().median()),
    }

    return priors_hour, priors_global

def make_deployment_style_row(ts, temp_c, priors_by_hour, priors_global):
    ts = pd.Timestamp(ts)
    hour = int(ts.hour)
    dayofweek = int(ts.dayofweek)
    minute_of_day = int(ts.hour * 60 + ts.minute)
    slot_5min = int(minute_of_day // 5)

    pri = priors_by_hour.get(hour, {})
    row = {
        "timestamp": ts,
        "power_w": np.nan,
        "temp_c": float(temp_c),
        "voltage_v": float(pri.get("voltage_v", priors_global["voltage_v"])),
        "amperage_a": float(pri.get("amperage_a", priors_global["amperage_a"])),
        "generation_w": float(pri.get("generation_w", priors_global["generation_w"])),
        "hour": hour,
        "dayofweek": dayofweek,
        "minute_of_day": minute_of_day,
        "slot_5min": slot_5min,
        "lag_5min": float(priors_global["lag_5min"]),
        "lag_10min": float(priors_global["lag_10min"]),
        "lag_1h": float(priors_global["lag_1h"]),
        "lag_24h": float(priors_global["lag_24h"]),
        "ma_1h": float(priors_global["ma_1h"]),
        "ma_3h": float(priors_global["ma_3h"]),
    }
    return row

def make_context_from_priors(ts, priors_by_hour, priors_global, seq_len):
    ts = pd.Timestamp(ts)
    rows = []
    for i in range(seq_len, 0, -1):
        ctx_ts = ts - pd.Timedelta(hours=i)
        hour = int(ctx_ts.hour)
        pri = priors_by_hour.get(hour, {})
        minute_of_day = int(ctx_ts.hour * 60 + ctx_ts.minute)
        slot_5min = int(minute_of_day // 5)

        rows.append({
            "timestamp": ctx_ts,
            "power_w": float(priors_global["lag_1h"]),
            "temp_c": float(priors_global.get("temp_c_fallback", 0.0)),
            "voltage_v": float(pri.get("voltage_v", priors_global["voltage_v"])),
            "amperage_a": float(pri.get("amperage_a", priors_global["amperage_a"])),
            "generation_w": float(pri.get("generation_w", priors_global["generation_w"])),
            "hour": hour,
            "dayofweek": int(ctx_ts.dayofweek),
            "minute_of_day": minute_of_day,
            "slot_5min": slot_5min,
            "lag_5min": float(priors_global["lag_5min"]),
            "lag_10min": float(priors_global["lag_10min"]),
            "lag_1h": float(priors_global["lag_1h"]),
            "lag_24h": float(priors_global["lag_24h"]),
            "ma_1h": float(priors_global["ma_1h"]),
            "ma_3h": float(priors_global["ma_3h"]),
        })
    return rows

def build_deployment_sequences(hourly_df, scaler_X, priors_hour_df, priors_global, seq_len=24):
    priors_by_hour = priors_hour_df.set_index("hour")[["voltage_v", "amperage_a", "generation_w"]].to_dict("index")

    X_list, ts_list, y_list = [], [], []
    for _, r in hourly_df.iterrows():
        ts = pd.Timestamp(r["timestamp"])
        temp_c = float(r["temp_c"])
        y_true = float(r["power_w"])

        context_rows = make_context_from_priors(ts, priors_by_hour, priors_global, seq_len - 1)
        current_row = make_deployment_style_row(ts, temp_c, priors_by_hour, priors_global)
        seq_df = pd.DataFrame(context_rows + [current_row])

        X_seq = seq_df[TRAIN_FEATURES].to_numpy(dtype=float).reshape(1, seq_len, len(TRAIN_FEATURES))
        X_sc = scale_sequence_tensor(X_seq, scaler_X)

        X_list.append(X_sc[0])
        ts_list.append(ts)
        y_list.append(y_true)

    return np.asarray(X_list), np.asarray(y_list), np.asarray(ts_list)

def build_route1_residual_buckets_from_val_asymmetric(val_df_5m, scaler_X, model, scaler_y, priors_hour, priors_global):
    val_h = aggregate_to_hourly(val_df_5m)

    X_val_dep, y_val_dep, ts_val_dep = build_deployment_sequences(
        hourly_df=val_h,
        scaler_X=scaler_X,
        priors_hour_df=priors_hour,
        priors_global=priors_global,
        seq_len=SEQ_LEN
    )

    yhat_sc = model.predict(X_val_dep, verbose=0)
    yhat_val = scaler_y.inverse_transform(yhat_sc).flatten()

    resid = y_val_dep - yhat_val
    clip_val = np.percentile(np.abs(resid), 99)
    resid = np.clip(resid, -clip_val, clip_val)

    df_tmp = pd.DataFrame({
        "timestamp": pd.to_datetime(ts_val_dep),
        "point_pred": yhat_val,
        "residual": resid,
    })
    df_tmp["hour"] = df_tmp["timestamp"].dt.hour
    df_tmp["dayofweek"] = df_tmp["timestamp"].dt.dayofweek

    res_q = (
        df_tmp.groupby(["hour", "dayofweek"])["residual"]
              .quantile([Q_LO, Q_MID, Q_HI])
              .unstack()
              .rename(columns={Q_LO: "rq10", Q_MID: "rq50", Q_HI: "rq90"})
              .reset_index()
    )

    sigma_global = float(np.std(resid))
    sigma_global = max(sigma_global, 1e-6)

    return res_q, sigma_global, df_tmp

def apply_route1_buckets_hourly(test_timestamps, point_pred, res_q):
    tmp = pd.DataFrame({"timestamp": pd.to_datetime(test_timestamps)})
    tmp["hour"] = tmp["timestamp"].dt.hour
    tmp["dayofweek"] = tmp["timestamp"].dt.dayofweek

    tmp = tmp.merge(res_q, on=["hour", "dayofweek"], how="left")
    tmp[["rq10", "rq50", "rq90"]] = tmp[["rq10", "rq50", "rq90"]].fillna(0.0)

    p10 = point_pred + tmp["rq10"].to_numpy()
    p50 = point_pred + tmp["rq50"].to_numpy()
    p90 = point_pred + tmp["rq90"].to_numpy()

    return enforce_non_crossing(p10, p50, p90)

# =========================
# 5) Load V1 and V2 at 5-minute resolution
# =========================
print("Loading V1 and V2 at 5-minute resolution...")
v1_5m = load_raw_multivar_5min(V1_PATH, V1_START_DATE, SRC_FREQ)
v2_5m = load_raw_multivar_5min(V2_PATH, V2_START_DATE, SRC_FREQ)

full_5m = pd.concat([v1_5m, v2_5m], ignore_index=True)
full_5m = full_5m.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)
full_5m = force_5min_grid(full_5m)
full_5m = add_engineered_features_5min(full_5m)

v1_5m = full_5m[full_5m["timestamp"] < pd.Timestamp(V2_START_DATE)].copy().reset_index(drop=True)
v2_5m = full_5m[full_5m["timestamp"] >= pd.Timestamp(V2_START_DATE)].copy().reset_index(drop=True)

print("✅ V1 5-min samples:", len(v1_5m))
print("✅ V2 5-min samples:", len(v2_5m))

# =========================
# 6) Split: validation / test
# =========================
test_mask = v2_5m["timestamp"].dt.normalize().isin(TEST_DAYS)

train_df_5m = v1_5m.copy().reset_index(drop=True)
val_df_5m = v2_5m[~test_mask].copy().reset_index(drop=True)
test_df_5m = v2_5m[test_mask].copy().reset_index(drop=True)

print("✅ Train 5-min samples:", len(train_df_5m))
print("✅ Val   5-min samples:", len(val_df_5m))
print("✅ Test  5-min samples:", len(test_df_5m))

test_df_hourly_all = aggregate_to_hourly(test_df_5m)
print("✅ Test hourly samples:", len(test_df_hourly_all))

for d in TEST_DAYS:
    cur = test_df_hourly_all[test_df_hourly_all["timestamp"].dt.normalize() == d].copy()
    if len(cur) != EXPECTED_HOURLY_POINTS_PER_DAY:
        raise ValueError(f"Test day {d.date()} has {len(cur)} hourly points, expected 24.")

# =========================
# 7) Build training priors for deployment-time missing variables
# =========================
print("\nBuilding V1 priors for deployment-time missing variables...")
priors_hour, priors_global = build_hourly_priors_from_train(train_df_5m)
priors_global["temp_c_fallback"] = float(train_df_5m["temp_c"].median())

pd.DataFrame(priors_hour).to_csv(os.path.join(OUT_DIR, "v1_hourly_priors_for_missing_vars.csv"), index=False)
pd.DataFrame([priors_global]).to_csv(os.path.join(OUT_DIR, "v1_global_priors_for_missing_vars.csv"), index=False)

# =========================
# 8) Train Route1 point model on V1 rich 5-minute multivariate
# =========================
print("\nTraining Route1 point BiGRU-LSTM on V1 rich 5-minute multivariate...")

split_train = int(0.90 * len(train_df_5m))
tr_df = train_df_5m.iloc[:split_train].copy()
va_df = train_df_5m.iloc[split_train:].copy()

if len(va_df) < max(SEQ_LEN + 24, 288):
    va_df = train_df_5m.iloc[-max(SEQ_LEN + 288, 576):].copy()
    tr_df = train_df_5m.iloc[:-len(va_df)].copy()

t0 = time.time()
point_model, point_sX, point_sY, hist_pt = train_point_model(tr_df, va_df)
print(f"Point model fit time: {time.time()-t0:.2f}s | train_rows={len(tr_df)} | val_rows={len(va_df)}")

# =========================
# 9) Train Route2 probabilistic model on V1 rich 5-minute multivariate
# =========================
print("\nTraining Route2 probabilistic BiGRU-LSTM on V1 rich 5-minute multivariate...")
t0 = time.time()
prob_model, prob_sX, prob_sY, hist_pr = train_prob_model(tr_df, va_df)
print(f"Prob model fit time: {time.time()-t0:.2f}s | train_rows={len(tr_df)} | val_rows={len(va_df)}")

# =========================
# 10) Build Route1 residual buckets from V2 validation
#     in deployment-style degraded hourly mode
# =========================
print("\nBuilding Route1 residual buckets from V2 validation in deployment-style hourly mode...")
res_q, sigma_global, df_res = build_route1_residual_buckets_from_val_asymmetric(
    val_df_5m=val_df_5m,
    scaler_X=point_sX,
    model=point_model,
    scaler_y=point_sY,
    priors_hour=priors_hour,
    priors_global=priors_global
)

resq_path = os.path.join(OUT_DIR, "bigrulstm_route1_residual_quantiles_from_v2val_asymmetric_hourly.csv")
resid_path = os.path.join(OUT_DIR, "bigrulstm_route1_residuals_from_v2val_asymmetric_hourly.csv")
res_q.to_csv(resq_path, index=False)
df_res.to_csv(resid_path, index=False)

print("✅ Saved residual buckets:", resq_path)
print("✅ Saved residual table  :", resid_path)
print(f"Fallback sigma_global: {sigma_global:.4f}")

# =========================
# 11) Test on Dec 24 and Dec 30 in degraded hourly mode
# =========================
print("\nTesting on Dec 24 and Dec 30 in degraded hourly mode...")

r1_uq_day_files = []
r2_prob_day_files = []

for k, day_start in enumerate(TEST_DAYS, start=1):
    print("\n" + "=" * 70)
    print(f"Test day {k}: {day_start.date()}")

    day_end = day_start + pd.Timedelta(days=1)

    test_df_h = test_df_hourly_all[
        (test_df_hourly_all["timestamp"] >= day_start) &
        (test_df_hourly_all["timestamp"] < day_end)
    ].copy().reset_index(drop=True)

    if len(test_df_h) != EXPECTED_HOURLY_POINTS_PER_DAY:
        raise ValueError(f"Test day {day_start.date()} does not have 24 hourly samples.")

    X_test_point, y_true, ts_test = build_deployment_sequences(
        hourly_df=test_df_h,
        scaler_X=point_sX,
        priors_hour_df=priors_hour,
        priors_global=priors_global,
        seq_len=SEQ_LEN
    )

    yhat_sc = point_model.predict(X_test_point, verbose=0)
    point_pred = point_sY.inverse_transform(yhat_sc).flatten()

    p10_uq, p50_uq, p90_uq = apply_route1_buckets_hourly(ts_test, point_pred, res_q)

    out_r1 = pd.DataFrame({
        "timestamp": ts_test,
        "actual": y_true,
        "point_pred": point_pred,
        "q10": p10_uq,
        "q50": p50_uq,
        "q90": p90_uq
    })
    p_r1 = os.path.join(OUT_DIR, f"bigrulstm_route1_uq_asymmetric_hourly_day{k}.csv")
    out_r1.to_csv(p_r1, index=False)
    r1_uq_day_files.append(p_r1)

    X_test_prob, _, ts_test_prob = build_deployment_sequences(
        hourly_df=test_df_h,
        scaler_X=prob_sX,
        priors_hour_df=priors_hour,
        priors_global=priors_global,
        seq_len=SEQ_LEN
    )

    yq_sc = prob_model.predict(X_test_prob, verbose=0)
    yq_sc = np.sort(yq_sc, axis=1)

    q10_prob = prob_sY.inverse_transform(yq_sc[:, [0]]).flatten()
    q50_prob = prob_sY.inverse_transform(yq_sc[:, [1]]).flatten()
    q90_prob = prob_sY.inverse_transform(yq_sc[:, [2]]).flatten()

    q10_prob, q50_prob, q90_prob = enforce_non_crossing(q10_prob, q50_prob, q90_prob)

    out_r2 = pd.DataFrame({
        "timestamp": ts_test_prob,
        "actual": y_true,
        "q10": q10_prob,
        "q50": q50_prob,
        "q90": q90_prob
    })
    p_r2 = os.path.join(OUT_DIR, f"bigrulstm_route2_prob_asymmetric_hourly_day{k}.csv")
    out_r2.to_csv(p_r2, index=False)
    r2_prob_day_files.append(p_r2)

    print("✅ Saved:")
    print("  Route1_UQ   :", p_r1)
    print("  Route2_Prob :", p_r2)

# =========================
# 12) Merge outputs
# =========================
print("\nMerging outputs...")

merged_route1_path = os.path.join(OUT_DIR, "merged_route1_point_uq_series_bigrulstm_asymmetric_hourly.csv")
merged_route1_df = merge_daily_route1(r1_uq_day_files, merged_route1_path)

merged_route2_path = os.path.join(OUT_DIR, "merged_route2_predictions_prob_bigrulstm_asymmetric_hourly.csv")
merged_route2_df = merge_daily_route2(r2_prob_day_files, merged_route2_path)

print("✅ Merged files written:")
print("  Route1:", merged_route1_path)
print("  Route2:", merged_route2_path)
print("  OUT_DIR:", OUT_DIR)

print("\nRoute1 head:")
print(merged_route1_df.head())

print("\nRoute2 head:")
print(merged_route2_df.head())